# TechStack: Data Preparation and ML Baseline

## 1. Project Overview

TechStack is designed as a scalable water-quality forecasting platform for multiple rivers. Because the prototype was developed under limited time, dataset availability, and computational constraints, this implementation uses historical Ganga River data as a baseline reference case.

This proof of concept validates the core data-to-model workflow before extending it to additional Indian rivers. The long-term architecture is:

**Multiple Rivers -> Standardized Data Pipeline -> Feature Engineering -> ML Models -> Water-Quality Forecasting**

The current implementation should therefore be read as a Ganga-based reference implementation, not as a model already trained across multiple rivers.

## 2. Environment Setup

### Import Libraries

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

### Configuration

In [2]:
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

### Project Root Detection

In [3]:
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
print("Project root:", ROOT)

Project root: C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack


### Define Paths

In [4]:
RAW_WATER = ROOT / "data" / "raw" / "water_quality"
RAW_WEATHER = ROOT / "data" / "raw" / "weather"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
FIGURES = ROOT / "images"

### Create Required Directories

In [5]:
MODELS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
print("Directories ready.")

Directories ready.


### Verify Environment

In [6]:
for required_path in [RAW_WATER, RAW_WEATHER, PROCESSED, MODELS, FIGURES]:
    print(f"{required_path}: {required_path.exists()}")

C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\data\raw\water_quality: True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\data\raw\weather: True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\data\processed: True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\models: True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\images: True


## 3. Data Loading

The water-quality and weather masters were prepared in the earlier data-cleaning workflow. Weather is loaded for inspection but is not merged into this historical ML baseline because date compatibility has not been established.

### Load Water Quality Data

In [7]:
water_quality = pd.read_csv(PROCESSED / "water_quality_master.csv")

### Load Weather Data

In [8]:
weather = pd.read_csv(PROCESSED / "weather_master.csv")

### Load Prepared ML Dataset

In [9]:
prepared_model = pd.read_csv(PROCESSED / "model_dataset.csv")

### Display Dataset Shapes

In [10]:
print("Water-quality master:", water_quality.shape)
print("Weather master:", weather.shape)
print("Prepared model dataset:", prepared_model.shape)

Water-quality master: (413, 7)
Weather master: (1152, 14)
Prepared model dataset: (326, 6)


### Preview Data

In [11]:
display(water_quality.head())
display(prepared_model.head())

,state,station_name,year,dissolved_oxygen,bod,fecal_coliform,source_file
0,Uttarakhand,GANGA AT HARIDWAR D/S,2011.0,6.7,5.6,1150.0,rs_session-241_as196_1.1.csv
1,Uttarakhand,GANGA AT HARIDWAR D/S,2012.0,7.2,5.3,NaN,rs_session-241_as196_1.1.csv
2,Uttarakhand,GANGA AT HARIDWAR D/S,2013.0,6.5,5.2,NaN,rs_session-241_as196_1.1.csv
3,Uttarakhand,GANGA AT HARIDWAR D/S,2014.0,5.0,5.2,NaN,rs_session-241_as196_1.1.csv
4,Uttarakhand,GANGA AT HARIDWAR D/S,2015.0,9.2,2.8,580.0,rs_session-241_as196_1.1.csv


,state,year,dissolved_oxygen,bod,fecal_coliform,source_file
0,Uttarakhand,2011,6.7,5.6,1150.0,rs_session-241_as196_1.1.csv
1,Uttarakhand,2012,7.2,5.3,NaN,rs_session-241_as196_1.1.csv
2,Uttarakhand,2013,6.5,5.2,NaN,rs_session-241_as196_1.1.csv
3,Uttarakhand,2014,5.0,5.2,NaN,rs_session-241_as196_1.1.csv
4,Uttarakhand,2015,9.2,2.8,580.0,rs_session-241_as196_1.1.csv


### Display Dataset Columns

In [12]:
print("Water-quality columns:", water_quality.columns.tolist())
print("Weather columns:", weather.columns.tolist())
print("Prepared model columns:", prepared_model.columns.tolist())

Water-quality columns: ['state', 'station_name', 'year', 'dissolved_oxygen', 'bod', 'fecal_coliform', 'source_file']
Weather columns: ['location', 'time', 'temperature', 'dew_point', 'humidity', 'precipitation', 'snow', 'wind_direction', 'wind_speed', 'wind_gust', 'pressure', 'sunshine_duration', 'weather_code', 'source_file']
Prepared model columns: ['state', 'year', 'dissolved_oxygen', 'bod', 'fecal_coliform', 'source_file']


## 4. Data Quality and Validation

### Check Missing Values

In [13]:
display(water_quality.isna().sum().rename("missing_count").to_frame())

,missing_count
state,0
station_name,0
year,84
dissolved_oxygen,3
bod,363
fecal_coliform,31
source_file,0


### Check Duplicate Records

In [14]:
print("Duplicate water-quality rows:", water_quality.duplicated().sum())
print("Duplicate prepared model rows:", prepared_model.duplicated().sum())

Duplicate water-quality rows: 0
Duplicate prepared model rows: 20


### Display Data Types

In [15]:
display(water_quality.dtypes.rename("dtype").to_frame())

,dtype
state,str
station_name,str
year,float64
dissolved_oxygen,float64
bod,float64
fecal_coliform,float64
source_file,str


### Basic Dataset Summary

In [16]:
display(water_quality.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
state,413,5,Uttar Pradesh,160,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station_name,413,107,GANGA AT HARIDWAR D/S,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,329.0,NaN,NaN,NaN,2018.088146,2.35069,2011.0,2018.0,2019.0,2020.0,2020.0
dissolved_oxygen,410.0,NaN,NaN,NaN,8.114976,1.175026,5.0,7.4125,8.1,8.9875,11.2
bod,50.0,NaN,NaN,NaN,4.47,1.373399,2.8,3.625,4.15,4.975,8.4
fecal_coliform,382.0,NaN,NaN,NaN,23847.180628,57571.671385,1.8,1300.0,6100.0,22000.0,592500.0
source_file,413,3,RS_Session_255_AU_90.1 (1).csv,279,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Exploratory Analysis

### Target Variable Distribution

In [17]:
target_available = water_quality.dropna(subset=["dissolved_oxygen"])
plt.figure(figsize=(8, 5))
plt.hist(target_available["dissolved_oxygen"], bins=20, color="#176b87", edgecolor="white")
plt.title("Dissolved Oxygen Distribution: Ganga Baseline Data")
plt.xlabel("Dissolved oxygen")
plt.ylabel("Record count")
plt.tight_layout()
plt.savefig(FIGURES / "01_target_distribution.png", dpi=160)
plt.show()

### Temporal Coverage

In [18]:
display(water_quality["year"].value_counts(dropna=False).sort_index().rename("record_count").to_frame())

,record_count
year,
2011.0,10
2012.0,10
2013.0,10
2014.0,10
2015.0,10
2018.0,93
2019.0,93
2020.0,93
NaN,84


## 6. Define the Machine Learning Problem

### Prediction Target

The target is `dissolved_oxygen`, treated as a continuous regression value.

### Input Features

The current baseline uses `state`, `station_name`, and `year`. These are the same features used by the existing workflow.

### Important Scope Note

This model is trained on the available Ganga River reference dataset. The pipeline is structured so future standardized river datasets can add a river identifier, river-specific stations, and river-aware feature engineering. This notebook does not claim multi-river training or generalization.

## 7. Train-Test Split

### Define Features

In [19]:
FEATURES = ["state", "station_name", "year"]
print("Features:", FEATURES)

Features: ['state', 'station_name', 'year']


### Define Target

In [20]:
TARGET = "dissolved_oxygen"
print("Target:", TARGET)

Target: dissolved_oxygen


### Create Training Dataset

In [21]:
model_data = water_quality[FEATURES + [TARGET, "source_file"]].dropna(subset=[TARGET, "year"]).copy()
model_data["year"] = model_data["year"].astype(int)
cutoff_year = model_data["year"].quantile(0.80)
train = model_data[model_data["year"] < cutoff_year].copy()
X_train = train[FEATURES]
y_train = train[TARGET]

### Create Testing Dataset

In [22]:
test = model_data[model_data["year"] >= cutoff_year].copy()
X_test = test[FEATURES]
y_test = test[TARGET]

### Display Split Information

In [23]:
print("Usable records:", len(model_data))
print("Excluded records:", len(water_quality) - len(model_data))
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training years:", sorted(train["year"].unique().tolist()))
print("Testing years:", sorted(test["year"].unique().tolist()))

Usable records: 326
Excluded records: 87
Training shape: (233, 3)
Testing shape: (93, 3)
Training years: [2011, 2012, 2013, 2014, 2015, 2018, 2019]
Testing years: [2020]


## 8. Feature Preprocessing

### Identify Categorical Features

In [24]:
categorical_features = ["state", "station_name"]
print(categorical_features)

['state', 'station_name']


### Identify Numerical Features

In [25]:
numerical_features = ["year"]
print(numerical_features)

['year']


### Categorical Preprocessing Pipeline

In [26]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

### Numerical Preprocessing Pipeline

In [27]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

### Combine Preprocessors

In [28]:
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numerical_pipeline, numerical_features),
])
print("Preprocessor ready.")

Preprocessor ready.


## 9. Model Definitions

### Linear Regression Definition

In [29]:
linear_regression = LinearRegression()

### Random Forest Definition

In [30]:
random_forest = RandomForestRegressor(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
)

### Gradient Boosting Definition

In [31]:
gradient_boosting = GradientBoostingRegressor(
    random_state=RANDOM_STATE, n_estimators=150, max_depth=2, learning_rate=0.04, loss="huber"
)

### Model Registry

In [32]:
model_registry = {
    "Linear Regression": linear_regression,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting,
}

## 10. Model Training

### Define Training Helper Function

In [33]:
def build_pipeline(estimator):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])

def train_model(estimator, name):
    fitted = build_pipeline(estimator)
    fitted.fit(X_train, y_train)
    print(f"{name} trained on {len(X_train)} records.")
    return fitted

### Train Linear Regression

In [34]:
linear_pipeline = train_model(linear_regression, "Linear Regression")

Linear Regression trained on 233 records.


### Train Random Forest

In [35]:
random_forest_pipeline = train_model(random_forest, "Random Forest")

Random Forest trained on 233 records.


### Train Gradient Boosting

In [36]:
gradient_boosting_pipeline = train_model(gradient_boosting, "Gradient Boosting")

Gradient Boosting trained on 233 records.


### Store Trained Pipelines

In [37]:
trained_pipelines = {
    "Linear Regression": linear_pipeline,
    "Random Forest": random_forest_pipeline,
    "Gradient Boosting": gradient_boosting_pipeline,
}

## 11. Model Predictions

### Predict with Linear Regression

In [38]:
linear_predictions = linear_pipeline.predict(X_test)

### Predict with Random Forest

In [39]:
random_forest_predictions = random_forest_pipeline.predict(X_test)

### Predict with Gradient Boosting

In [40]:
gradient_boosting_predictions = gradient_boosting_pipeline.predict(X_test)

### Store Predictions

In [41]:
predictions = {
    "Linear Regression": linear_predictions,
    "Random Forest": random_forest_predictions,
    "Gradient Boosting": gradient_boosting_predictions,
}

## 12. Model Evaluation

### Define Evaluation Function

In [42]:
def evaluate_predictions(actual, predicted, model_name):
    return {
        "model": model_name,
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "r2": r2_score(actual, predicted),
    }

### Evaluate Linear Regression

In [43]:
linear_metrics = evaluate_predictions(y_test, linear_predictions, "Linear Regression")
print(linear_metrics)

{'model': 'Linear Regression', 'mae': 0.48781470582039177, 'rmse': np.float64(0.6049026455534323), 'r2': 0.7300306874558002}


### Evaluate Random Forest

In [44]:
random_forest_metrics = evaluate_predictions(y_test, random_forest_predictions, "Random Forest")
print(random_forest_metrics)

{'model': 'Random Forest', 'mae': 0.49458915994335234, 'rmse': np.float64(0.6282060074524558), 'r2': 0.7088293451535654}


### Evaluate Gradient Boosting

In [45]:
gradient_boosting_metrics = evaluate_predictions(y_test, gradient_boosting_predictions, "Gradient Boosting")
print(gradient_boosting_metrics)

{'model': 'Gradient Boosting', 'mae': 0.5992716295183023, 'rmse': np.float64(0.7395098215336472), 'r2': 0.5965113474466548}


### Create Model Comparison Table

In [46]:
results = pd.DataFrame([linear_metrics, random_forest_metrics, gradient_boosting_metrics])
results = results.sort_values(["rmse", "mae"]).reset_index(drop=True)
display(results)

,model,mae,rmse,r2
0,Linear Regression,0.487815,0.604903,0.730031
1,Random Forest,0.494589,0.628206,0.708829
2,Gradient Boosting,0.599272,0.739510,0.596511


## 13. Model Comparison and Selection

The selected model is the model with the lowest RMSE on the chronological holdout, using MAE as a tie-breaker. This choice describes performance on the current Ganga baseline dataset only. It is not a claim that the same model will be best for every river or future data release.

In [47]:
best_model_name = results.iloc[0]["model"]
best_pipeline = trained_pipelines[best_model_name]
best_predictions = predictions[best_model_name]
print("Selected model on this baseline:", best_model_name)

Selected model on this baseline: Linear Regression


## 14. Diagnostics and Visualization

### Model Comparison Chart

In [48]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric in zip(axes, ["mae", "rmse", "r2"]):
    ordered = results.sort_values(metric, ascending=(metric != "r2"))
    ax.bar(ordered["model"], ordered[metric], color=["#176b87", "#e07a5f", "#3a7d44"])
    ax.set_title(metric.upper())
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Chronological Holdout Model Comparison: Ganga Baseline")
fig.tight_layout()
fig.savefig(FIGURES / "02_model_comparison.png", dpi=160)
plt.show()

### Actual vs Predicted

In [49]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, best_predictions, alpha=0.75, color="#176b87")
lo = min(y_test.min(), best_predictions.min())
hi = max(y_test.max(), best_predictions.max())
ax.plot([lo, hi], [lo, hi], "--", color="#e07a5f", label="Perfect prediction")
ax.set_title(f"Actual vs Predicted Dissolved Oxygen: {best_model_name}")
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "03_actual_vs_predicted.png", dpi=160)
plt.show()

### Residual Plot

In [50]:
residuals = y_test.to_numpy() - best_predictions
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_predictions, residuals, alpha=0.75, color="#3a7d44")
ax.axhline(0, linestyle="--", color="#e07a5f")
ax.set_title(f"Residual Diagnostics: {best_model_name}")
ax.set_xlabel("Predicted dissolved oxygen")
ax.set_ylabel("Residual (actual - predicted)")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "04_residual_diagnostics.png", dpi=160)
plt.show()

## 15. Save Results and Artifacts

### Save Model Comparison Results

In [51]:
results.to_csv(PROCESSED / "model_comparison.csv", index=False)

### Save Best Model and Supporting Pipelines

In [52]:
for name, pipeline in trained_pipelines.items():
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(pipeline, MODELS / f"{safe_name}_water_quality.joblib")
joblib.dump(best_pipeline, MODELS / "best_water_quality_model.joblib")

['C:\\Users\\AARYA PATEL\\OneDrive\\Desktop\\Github\\TechStack\\models\\best_water_quality_model.joblib']

### Save Predictions

In [53]:
prediction_output = test[["state", "station_name", "year", TARGET, "source_file"]].copy()
prediction_output["prediction"] = best_predictions
prediction_output["residual"] = residuals
prediction_output.to_csv(PROCESSED / "best_model_predictions.csv", index=False)

### Save Metadata

In [54]:
metadata = {
    "project_scope": "Ganga River baseline/reference implementation",
    "target": TARGET,
    "features": FEATURES,
    "models_compared": list(trained_pipelines),
    "selected_model": best_model_name,
    "train_years": sorted(train["year"].unique().tolist()),
    "test_years": sorted(test["year"].unique().tolist()),
    "metrics": results.to_dict(orient="records"),
    "limitations": [
        "The current model is trained on the available Ganga baseline data, not multiple rivers.",
        "Weather was not merged because historical date compatibility is unestablished.",
        "Satellite features are excluded from the initial model.",
        "Synthetic IoT readings are not historical ground truth.",
    ],
}
(MODELS / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

1219

### Verify Saved Files

In [55]:
for output_path in [
    PROCESSED / "model_comparison.csv",
    PROCESSED / "best_model_predictions.csv",
    MODELS / "best_water_quality_model.joblib",
    MODELS / "model_metadata.json",
    FIGURES / "01_target_distribution.png",
    FIGURES / "02_model_comparison.png",
    FIGURES / "03_actual_vs_predicted.png",
    FIGURES / "04_residual_diagnostics.png",
]:
    print(output_path, output_path.exists())

C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\data\processed\model_comparison.csv True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\data\processed\best_model_predictions.csv True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\models\best_water_quality_model.joblib True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\models\model_metadata.json True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\images\01_target_distribution.png True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\images\02_model_comparison.png True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\images\03_actual_vs_predicted.png True
C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack\images\04_residual_diagnostics.png True


## 16. Current Limitations

- The current prototype uses Ganga River data as the baseline reference case.
- Standardized multi-river datasets are not yet available in this workflow.
- River-specific environmental characteristics may affect model performance.
- More historical data is required to assess true multi-river generalization.
- Results should be interpreted as baseline prototype results, not universal performance claims.
- Weather remains separate until compatible historical dates are established.
- Satellite data is not included in the initial ML model.

## 17. Future Multi-River Roadmap

1. Integrate datasets from additional Indian rivers.
2. Add a river identifier to standardized records.
3. Standardize stations and monitoring data across rivers.
4. Develop river-aware feature engineering.
5. Evaluate generalized versus river-specific models.
6. Expand forecasting targets.
7. Build larger historical datasets and stronger validation periods.

## 18. Final Summary

A working ML baseline has been developed for the TechStack proof of concept. Ganga River data is used as the current reference implementation, and the notebook demonstrates the core data loading, validation, preprocessing, training, evaluation, diagnostics, and artifact-saving workflow. The architecture remains intended for future expansion to multiple rivers; this notebook does not claim that multi-river training has already been completed.